# Electrical (Lissajous) metric fits vs residence time / SEI

Consumes the **run-level** combined file produced by
`concat_electrical_calculations.ipynb`
(`uhasselt_co2_spl_electrical_data_combined.parquet`, one row per reactor run,
each metric `X` with a between-snapshot `X_sd`).

For every material and every electrical metric we fit a **simple linear** model
`X ~ residence_time_s`, weighted by `1 / X_sd**2` (the run measurement error).
A linear model is used deliberately: the true X-vs-residence-time relationship is
unknown, so a more complex form has no scientific basis. The ideal SEI is
proportional to residence time, so a line in residence time is also a line in SEI.

Output: `uhasselt_co2_spl_electrical_fitting_results.parquet`, **long over
`variable`**, with both the fitted grid (`row_type == 'fit'`: line + 95% band +
R2/slope/n) and the measured run-level points (`row_type == 'measured'`:
`value_measured` + `value_measured_sd`, plotted **without** error bars). All
fit-quality columns are repeated on every row of a (material, variable) group.

In [ ]:
import os
import numpy as np
import pandas as pd
from scipy.stats import t as student_t

parent_folder = r'C:\Users\Sande\OneDrive - Universiteit Antwerpen\plasmacatdesign\co2-splitting\uhasselt'
in_path = os.path.join(parent_folder, 'uhasselt_co2_spl_electrical_data_combined.parquet')
out_path = os.path.join(parent_folder, 'uhasselt_co2_spl_electrical_fitting_results.parquet')


def sei_kjmol(residence_time_s):
    """Ideal specific energy input (kJ/mol) for a residence time (s).

    Proportional to residence time, so it is purely an alternative x-axis; the
    linear fit is performed in residence time and SEI is attached for plotting.
    """
    return 30 / ((1 - 0.4687) * (17.31 / residence_time_s)) * 24.055


def fit_weighted_line(x, y, w):
    """Weighted least-squares straight line ``y = a + b*x`` (numpy/scipy only).

    Weights ``w`` are ``1 / measurement-variance`` but treated as RELATIVE: the
    band is scaled by the residual spread with ``df = n - 2`` (matching the
    previous statsmodels WLS harmonisation, and honest when the line does not
    perfectly fit). Returns the slope, its standard error, weighted R-squared,
    ``df``, ``n``, and a ``predict(xg) -> (mean, se_mean)`` closure where
    ``se_mean`` is the standard error of the predicted mean.
    """
    x = np.asarray(x, float)
    y = np.asarray(y, float)
    r = np.sqrt(np.asarray(w, float))
    X = np.column_stack([np.ones_like(x), x])
    Xw = X * r[:, None]                       # weighted design / response
    yw = y * r
    XtX_inv = np.linalg.inv(Xw.T @ Xw)
    beta = XtX_inv @ (Xw.T @ yw)
    df = len(x) - 2
    resid_w = yw - Xw @ beta
    cov = (resid_w @ resid_w) / df * XtX_inv  # scaled parameter covariance

    wmean = np.sum(w * y) / np.sum(w)
    ssr = np.sum(w * (y - (beta[0] + beta[1] * x)) ** 2)
    tss = np.sum(w * (y - wmean) ** 2)
    r2 = 1 - ssr / tss if tss > 0 else np.nan

    def predict(xg):
        Xg = np.column_stack([np.ones_like(np.asarray(xg, float)), np.asarray(xg, float)])
        mean = Xg @ beta
        se = np.sqrt(np.einsum('ij,jk,ik->i', Xg, cov, Xg))
        return mean, se

    return {
        'slope': beta[1], 'intercept': beta[0], 'slope_se': np.sqrt(cov[1, 1]),
        'r2': r2, 'df': df, 'n': len(x), 'predict': predict,
    }


def fit_weighted_constant(x, y, w):
    """Weighted intercept-only (slope fixed at 0) model ``y = a`` (constant).

    Used when a (material, variable) group has only 2 runs, where a straight line
    would have ``df = 0`` (perfect, no honest CI). The two residence times are too
    close to support a slope, so we fit a horizontal line: the weighted mean
    ``a = sum(w*y) / sum(w)`` with ``df = n - 1``. The band is scaled by the
    residual spread exactly like ``fit_weighted_line`` (weights RELATIVE), giving
    a genuine -- if wide -- confidence band. Returns the same dict shape so the
    caller is uniform; ``slope`` and ``slope_se`` are 0 by construction and ``r2``
    is NaN (against a weighted-mean baseline a constant model has R-squared == 0,
    which would be misleading). ``predict(xg)`` returns the flat mean and a
    constant standard error of that mean.
    """
    x = np.asarray(x, float)
    y = np.asarray(y, float)
    w = np.asarray(w, float)
    sw = np.sum(w)
    a = np.sum(w * y) / sw                     # weighted mean (the intercept)
    df = len(x) - 1
    var_a = np.sum(w * (y - a) ** 2) / df / sw  # scaled variance of the mean
    se_a = np.sqrt(var_a)

    def predict(xg):
        xg = np.asarray(xg, float)
        return np.full_like(xg, a), np.full_like(xg, se_a)

    return {
        'slope': 0.0, 'intercept': a, 'slope_se': 0.0,
        'r2': np.nan, 'df': df, 'n': len(x), 'predict': predict,
    }

In [ ]:
data = pd.read_parquet(in_path)

x_col = 'residence_time_s'
id_keys = [
    'project_name', 'reaction_type', 'material_supplier', 'material_name',
    'wattage_const', 'plasma_state',
]
non_metric = set(id_keys) | {x_col, 'measurement_number', 'n_snapshots'}

# Metric variables = numeric columns that are not identity/x and not the _sd pair.
variables = [
    c for c in data.columns
    if c not in non_metric and not c.endswith('_sd')
    and pd.api.types.is_numeric_dtype(data[c])
]
print(f'{len(variables)} electrical variables to fit across '
      f"{data['material_name'].nunique()} materials.")

In [ ]:
grid = np.arange(1.0, 100.5, 0.5)
MIN_POINTS = 3                      # >= 3 runs -> weighted line (df_resid >= 1)
frames = []

for packing, g in data.groupby('material_name'):
    for var in variables:
        sd_col = f'{var}_sd'
        cols = [x_col, 'measurement_number', var] + ([sd_col] if sd_col in g else [])
        sub = g[cols].dropna(subset=[x_col, var]).copy()
        if len(sub) < 2:            # need >= 2 runs for any fit with a CI
            continue

        x = sub[x_col].to_numpy(dtype=float)
        y = sub[var].to_numpy(dtype=float)

        # Weights = 1 / measurement-variance. Invalid SDs (<=0, NaN, or the whole
        # column missing) are replaced by the median valid SD so the point keeps a
        # finite, sensible weight; if none are valid we fall back to equal weights.
        if sd_col in sub:
            sd = sub[sd_col].to_numpy(dtype=float)
            valid = np.isfinite(sd) & (sd > 0)
            if valid.any():
                sd = np.where(valid, sd, np.median(sd[valid]))
                w = 1.0 / np.square(sd)
            else:
                w = np.ones_like(y)
        else:
            w = np.ones_like(y)

        # >= 3 runs: weighted line. Exactly 2 runs (e.g. recuperated data with two
        # close residence times): a slope is unjustified, so fit a constant
        # (slope=0) weighted model with an honest df = n - 1 = 1 band.
        if len(sub) >= MIN_POINTS:
            fit, fit_type = fit_weighted_line(x, y, w), 'linear'
        else:
            fit, fit_type = fit_weighted_constant(x, y, w), 'constant'

        tval = student_t.ppf(0.975, df=fit['df'])
        mean, se = fit['predict'](grid)
        slope_lo = fit['slope'] - tval * fit['slope_se']
        slope_hi = fit['slope'] + tval * fit['slope_se']

        meta = {
            'material_name': packing,
            'variable': var,
            'fit_type': fit_type,
            'r_squared': fit['r2'],
            'n_points': fit['n'],
            'df_resid': float(fit['df']),
            'slope': fit['slope'],
            'slope_ci_lo': slope_lo,
            'slope_ci_hi': slope_hi,
            'slope_ci_includes_zero': bool(slope_lo <= 0 <= slope_hi),
        }

        fit_rows = pd.DataFrame({
            'residence_time_s': grid,
            'sei_kjmol': sei_kjmol(grid),
            'row_type': 'fit',
            'value_fit': mean,
            'value_fit_stderr': se,
            'value_fit_lcl': mean - tval * se,
            'value_fit_ucl': mean + tval * se,
        })

        meas_rows = pd.DataFrame({
            'residence_time_s': sub[x_col].to_numpy(),
            'sei_kjmol': sei_kjmol(sub[x_col].to_numpy()),
            'row_type': 'measured',
            'measurement_number': sub['measurement_number'].to_numpy(),
            'value_measured': y,
            'value_measured_sd': (sub[sd_col].to_numpy() if sd_col in sub else np.nan),
        })

        block = pd.concat([fit_rows, meas_rows], ignore_index=True)
        for k, v in meta.items():
            block[k] = v
        frames.append(block)

results = pd.concat(frames, ignore_index=True)
print(f'electrical fit results: {results.shape}; '
      f"{results[['material_name','variable']].drop_duplicates().shape[0]} (material, variable) fits.")

In [ ]:
ordered = [
    'material_name', 'variable', 'residence_time_s', 'sei_kjmol', 'row_type',
    'value_fit', 'value_fit_stderr', 'value_fit_lcl', 'value_fit_ucl',
    'value_measured', 'value_measured_sd', 'measurement_number',
    'fit_type', 'r_squared', 'n_points', 'df_resid',
    'slope', 'slope_ci_lo', 'slope_ci_hi', 'slope_ci_includes_zero',
]
results = results[ordered]
results.to_parquet(out_path, index=False, compression='gzip')
print(f'Wrote {out_path}')

In [ ]:
%reset -f